In [ ]:
"""
Assignment Q1: n-qubit VQA (modified from 1-qubit warm-up)
-----------------------------------------------------------
Roll Number : 136
Number of Gates = n = 2  (one RY gate per qubit)

Goal  : Train parameters [00, 01] so that the 2-qubit ansatz
        RY(θ0)⊗RY(θ1)|00> minimises the cost w.r.t. the
        2-qubit Pauli-Z string observable ZZ.

Step 1: n_qubits = 2,  n_gates = 2
Step 2: Parameterised circuit — one RY gate on each qubit
Step 3: Measure <ZZ> over 1024 shots (via Estimator)
"""

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

N_QUBITS  = 2        # = 2
N_GATES   = 2              # = 2  (one RY per qubit)
N_SHOTS   = 1024

print(f"n_qubits    : {N_QUBITS}")
print(f"n_gates     : {N_GATES}")
print(f"shots       : {N_SHOTS}")

# ============================================================
# Step 2 — Parameterised ansatz circuit
# ============================================================
theta = ParameterVector("θ", N_QUBITS)   # [θ0, θ1]

qc = QuantumCircuit(N_QUBITS)
for i in range(N_QUBITS):
    qc.ry(theta[i], i)                   # Gate i : RY(θi) on qubit i

print("\nAnsatz circuit:")
print(qc.draw())

# ============================================================
# Step 3 — Observable: n-qubit Pauli-Z string  (ZZ for n=2)
#  
# ============================================================
pauli_str = "Z" * N_QUBITS              # "ZZ"
observable = SparsePauliOp(pauli_str)
print(f"\nObservable  : {pauli_str}")

estimator = StatevectorEstimator()

# Helper: evaluate <ZZ> for a given parameter array
def expval(params):
    """Return <ZZ> for RY(params[0])⊗RY(params[1])|00>."""
    param_values = list(params)
    job = estimator.run([(qc, observable, param_values)])
    return float(job.result()[0].data.evs)

# Sanity check:  <ZZ> = cos(θ0)*cos(θ1)  analytically
print("\nSanity check  (circuit vs analytic cos(θ0)*cos(θ1)):")
test_cases = [
    [0.0,      0.0],
    [np.pi/2,  0.0],
    [0.0,      np.pi/2],
    [np.pi/2,  np.pi/2],
    [np.pi,    np.pi],
]
for tc in test_cases:
    circuit_val = expval(tc)
    analytic    = np.cos(tc[0]) * np.cos(tc[1])
    print(f"  θ={tc}  circuit={circuit_val:+.4f}  analytic={analytic:+.4f}")

# ============================================================
# Cost function  &  parameter-shift gradients
# ============================================================
TARGET = -0.5

def cost(params):
    return (expval(params) - TARGET) ** 2

def param_shift_grad(params, s=np.pi / 2):
    """Parameter-shift gradient for each parameter."""
    grad = np.zeros(len(params))
    for i in range(len(params)):
        p_plus  = params.copy(); p_plus[i]  += s
        p_minus = params.copy(); p_minus[i] -= s
        d_expval = (expval(p_plus) - expval(p_minus)) / 2
        grad[i]  = 2 * (expval(params) - TARGET) * d_expval
    return grad

# ============================================================
# Training loop — gradient descent
# ============================================================
params = np.array([0.1, 0.2])     # initial parameters
lr     = 0.4
steps  = 40
history = {"params": [], "cost": [], "expval": []}

print(f"\nTraining loop  (TARGET <ZZ> = {TARGET})")
for step in range(steps):
    g      = param_shift_grad(params)
    params = params - lr * g
    e      = expval(params)
    c      = (e - TARGET) ** 2
    history["params"].append(params.copy())
    history["expval"].append(e)
    history["cost"].append(c)
    if step % 5 == 0:
        print(f"  step {step:2d}  params={np.round(params,4)}  "
              f"<ZZ>={e:+.4f}  cost={c:.2e}")

print(f"\nFinal params : {np.round(params, 4)} rad")
print(f"Final <ZZ>   : {expval(params):+.4f}  (target = {TARGET})")

# ============================================================
# Step 3 — Measure using 1024 shots (Sampler / counts)
#           and print the Pauli-Z string output
# ============================================================
from qiskit.primitives import StatevectorSampler

# Bind optimised parameters and add measurement
qc_meas = qc.assign_parameters(dict(zip(theta, params)))
qc_meas.measure_all()

sampler = StatevectorSampler()
job_s   = sampler.run([qc_meas], shots=N_SHOTS)
counts  = job_s.result()[0].data.meas.get_counts()

print(f"\nMeasurement results ({N_SHOTS} shots):")
for bitstring, count in sorted(counts.items(), key=lambda x: -x[1]):
    pauli_z_out = "".join("Z+" if b == "0" else "Z-" for b in bitstring)
    print(f"  |{bitstring}>  →  Pauli-Z string: {pauli_z_out}   counts: {count}")

# ============================================================
# Plots
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"2-Qubit VQA |  Qubits: {N_QUBITS}  |  Gates: {N_GATES}",
             fontsize=13)

# 1. Cost landscape (θ1 fixed at final value, sweep θ0)
grid = np.linspace(0, 2 * np.pi, 200)
landscape = (np.cos(grid) * np.cos(params[1]) - TARGET) ** 2
theta0_hist = [h[0] for h in history["params"]]
ax[0].plot(grid, landscape, label="cost landscape (θ1 fixed)")
ax[0].plot(theta0_hist, history["cost"], "o-", ms=3, label="training path")
ax[0].set_xlabel("θ0 (rad)"); ax[0].set_ylabel("cost")
ax[0].set_title("Cost vs θ0"); ax[0].legend(fontsize=8)

# 2. Convergence
ax[1].semilogy(np.maximum(history["cost"], 1e-30))
ax[1].set_xlabel("step"); ax[1].set_ylabel("cost (log)")
ax[1].set_title("Convergence")

# 3. <ZZ> during training
ax[2].plot(history["expval"], label="<ZZ>")
ax[2].axhline(TARGET, color="r", ls="--", label=f"target = {TARGET}")
ax[2].set_xlabel("step"); ax[2].legend()
ax[2].set_title("<ZZ> during training")

plt.tight_layout()
plt.savefig("vqa_2qubits.png", dpi=150)
plt.show()
print("\nPlot saved as vqa_2qubits.png")


: 